Structured output <break>
<break>
Models can be requested to provide their responses in a format matching a given schema


In [1]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

In [5]:
model = init_chat_model('groq:qwen/qwen3-32b')

### Pydantic<break>
pydantic models provide the richest feature set with field validation, descriptions, and nested structures

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title ofv the movie")
    year:int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    ratings: float = Field(description="The movie's ratings")

In [7]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002838962A330>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002838AC72ED0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title ofv the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'ratings': {'description': "The movie's ratings

In [8]:
model_with_structure.invoke("Provide me the details about the movie Vanilla sky")

Movie(title='Vanilla Sky', year=2001, director='Cameron Crowe', ratings=7.5)

In [ ]:
class Biography(BaseModel):
    name:str = Field(description="Name of the person")
    age:int = Field(description="Age of the person")
    nationality:str = Field(description="Nationality of the person")
    father_name: str = Field(description="Father's name")
    mother_name: str = Field(description="Mother's name")
    profession: str = Field(description="Profession of the person")

In [13]:
model_with_bio_structure = model.with_structured_output(Biography)

In [15]:
model_with_bio_structure.invoke("Who is salman khan?")

Biography(name='Salman Khan', age=58, nationality='Indian', father_name='Salim Khan', mother_name='Saba Khan', profession='Actor, producer, and philanthropist')

Message output along with parsed structure

In [19]:
class Biography(BaseModel):
    name:str = Field("Name of the person")
    age:int = Field("Age of the person")
    nationality:str = Field("Nationality of the person")
    father_name: str = Field("Father's name")
    mother_name: str = Field("Mother's name")
    profession: str = Field("Profession of the person")

model_with_bio_structure = model.with_structured_output(Biography, include_raw=True)


In [20]:
model_with_bio_structure.invoke("Who is salman khan?")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking "Who is Salman Khan?" I need to provide a biography of him. Let me recall the available function. The Biography function has parameters like name, age, father_name, mother_name, nationality, and profession. \n\nFirst, I know Salman Khan is a well-known Indian actor. His full name is Salman Khan. His age might be around 55, but I should verify that. His father is Salim Khan, and his mother is Sushila Khan. He\'s been in the film industry for a long time, so his profession is actor and producer. Nationality is Indian. \n\nWait, the age parameter is an integer. Let me check his birth year. He was born on December 27, 1965. So as of 2023, he\'s 57 years old. But since the user might be asking at any time, maybe I should use the current year minus 1965. But since the function expects a default value, maybe just input 58? Or does the function calculate it? The default is "Age of the person", but I

### Nested structure

In [24]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    age: int

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genred: list[str]
    budget: float | None = Field(None, description="Budget of the movie in USD")

model_wih_movie_structure = model.with_structured_output(MovieDetails)

In [27]:
response = model_wih_movie_structure.invoke("Vanilla sky")
print(response)

title='Vanilla Sky' year=2001 cast=[Actor(name='Tom Cruise', age=37), Actor(name='Cameron Diaz', age=28), Actor(name='Jason Lee', age=30)] genred=['Drama', 'Science Fiction'] budget=None


### TypeDict